# Silver ETL Job (Raw → Silver)
This notebook contains the finalized ETL logic used to transform Raw logs into the Silver dataset.
It reads the previous day's raw files from S3, normalizes schema, performs data quality checks,
and writes partitioned Silver data back to S3.

In [ ]:
import sys
from datetime import datetime, timedelta

from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql.functions import col, split


## Parameters
Reads previous day's raw data from S3.

In [ ]:
TARGET_DATE = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
RAW_PATH = f"s3://ameowzon-raw/raw/date={TARGET_DATE}/"
SILVER_PATH = "s3://ameowzon-silver/events/"

print(f"[INFO] TARGET_DATE = {TARGET_DATE}")
print(f"[INFO] RAW_PATH = {RAW_PATH}")

## Spark / Glue Context

In [ ]:
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

# overwrite 시 partition만 교체
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

## Read Raw Data

In [ ]:
raw_df = spark.read.json(RAW_PATH)
print("[INFO] Raw schema")
raw_df.printSchema()

## Normalize + Transform to Silver

In [ ]:
silver_df = (
    raw_df
    .withColumnRenamed("user_token", "uuid")
    .withColumnRenamed("package_token", "app_id")

    .withColumn(
        "event_name",
        F.when(F.col("type") == "heartbeat", "HEARTBEAT")
        .when(F.col("_activity") == "WIFI_CHANGE", "WIFI_SSID")
        .when(F.col("_activity") == "NOTIFICATION_INTERRUPTION", "NOTIF_INTERRUPTION")
        .otherwise(F.col("_activity"))
    )

    .withColumn("uuid", F.trim("uuid"))
    .withColumn("app_id", F.trim("app_id"))
    .withColumn("type", F.trim("type"))
    .withColumn("event_name", F.trim("event_name"))

    .withColumn("lat", split(col("coarse_gps"), ",").getItem(0).cast("double"))
    .withColumn("lon", split(col("coarse_gps"), ",").getItem(1).cast("double"))

    .withColumn(
        "cell_lac",
        F.when(col("cell_lac") == "unknown", None)
        .otherwise(col("cell_lac").cast("int"))
    )

    .withColumn(
        "prev_bssid",
        F.when(col("_activity") == "WIFI_CHANGE", col("prev_bssid"))
    )
    .withColumn(
        "curr_bssid",
        F.when(col("_activity") == "WIFI_CHANGE", col("curr_bssid"))
    )

    .withColumn(
        "local_ts",
        col("timestamp") + col("tz_offset_minutes") * 60000
    )
    .withColumn(
        "dt",
        F.to_date(F.from_unixtime(col("local_ts")/1000))
    )

    .dropDuplicates(["uuid","timestamp","event_name"])

    .select(
        "uuid",
        "event_name",
        "timestamp",
        "type",
        "app_id",
        "lat",
        "lon",
        "step_count",
        "queue_size",
        "retry_count",
        "client_last_event_ts",
        "tz_offset_minutes",
        "cell_lac",
        "prev_bssid",
        "curr_bssid",
        "dt"
    )
)

print("[INFO] Silver schema")
silver_df.printSchema()

## Data Quality Checks

In [ ]:
print("[CHECK] uuid whitespace:", silver_df.filter(F.col("uuid") != F.trim("uuid")).count())

print("[CHECK] timestamp null:", silver_df.filter(F.col("timestamp").isNull()).count())

silver_df.selectExpr(
    "count(*) as total",
    "sum(case when lat is null then 1 else 0 end) as lat_null",
    "sum(case when lon is null then 1 else 0 end) as lon_null"
).show()

silver_df.groupBy("type").count().show()

silver_df.groupBy("event_name") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(20, truncate=False)

## Write Silver Data

In [ ]:
(
    silver_df
    .repartition(4)
    .write
    .mode("overwrite")
    .partitionBy("dt")
    .parquet(SILVER_PATH)
)

print("[INFO] WRITE completed.")